# 21 Grid noise in the T-web basins

How much of the T-web basin merge tree is set by where the grid points happen to fall? The question behind it: SMDPL's true catalog has ~40% more voids per volume than TNG's or Bolshoi's, its labels are on 0.78 h⁻¹ Mpc cells instead of 0.40, and the basin settings (Δ, the minimum radius) are in h⁻¹ Mpc. If the noise in the tree grows with the cell, Δ should be set in units of that noise rather than as a fixed length.

**The test** (the grid version of Neyrinck's 2008 suggestion for ZOBOV: jitter the particles, rerun, and see which zones survive). For each density grid:

1. T-web labels (λ_th = 0.65, σ_eff = 1 h⁻¹ Mpc) on the grid as it is, and again with the grid points moved by half a cell, along x and along the diagonal. The move is a Fourier phase on the smoothed density, so it is an exact interpolation of the same field (`tweb_labels(..., shift=...)`).
2. For each: the distance to structure, its basins and the full merge tree, as in notebook 18.
3. Every merge of the unshifted tree joins two void centers (the peak basins of the two groups). Each center is found in the shifted grid (the shifted basin at the nearest grid point), and for both grids the **join height** is computed: the smallest Δ at which the two centers end up in one void, the same rule as `cut_tree` (`voids.join_heights`). Its change between the grids is grid noise; the real structure is the same.

The same is done with a scale-free measure, the ratio of the child's peak to the pass (ZOBOV's significance is such a ratio, of densities), to see whether it is steadier across grids.

**Grids.** TNG at 512³ (0.40 h⁻¹ Mpc) and 256³ (0.80, the 2×2×2 block mean of the 512³ density) and SMDPL at 512³ (0.78). If the noise at TNG 256³ is about twice that at 512³ and SMDPL's matches TNG 256³, the noise scales with the cell. The void counts above R = 3 h⁻¹ Mpc at Δ = 0.5, 1, 2 are given for each grid and shift as well.

**Output.** `void_definitions/grid_noise/<grid>_<shift>.npz` (each run's merge tree and the lookups; reused on reruns) and `void_definitions/grid_noise.json` (the tables), plus the figure `grid_noise.png`.

**Runtime.** CPU, High-RAM (the 512³ T-web needs ~6 GB, the distance transform more). Nine runs (three grids × three placements); each prints how long it took last time, if known.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

## 1 Settings

In [ ]:
import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator, ScalarFormatter

from deepvoid import Catalog
from deepvoid import voids as V
from deepvoid.io import load_volume
from deepvoid.labels import VOID, tweb_labels
from deepvoid.volume import Volume

RUNS = Path('/content/drive/MyDrive/deepvoid_runs')
OUT = RUNS / 'void_definitions'
RUN_DIR = OUT / 'grid_noise'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT = OUT / 'grid_noise.json'
TIMINGS = RUNS / 'voids' / 'timings.json'   # shared with notebooks 17 and 18
catalog = Catalog()

SMDPL = Path('/content/drive/MyDrive/data/SMDPL')
SMDPL_DENSITY = SMDPL / 'smdpl_dens512_z0.npy'               # notebook 15
SMDPL_LABELS = SMDPL / 'smdpl_mask_tweb_th0.65_seff1.npz'    # notebook 16, for a check
SMDPL_BOX = 400.0                                            # h^-1 Mpc
GRIDS = {   # name: (density, labels already in use, for a check)
    'TNG 512³ (0.40)': ('tng300-3/dm', 'tng300-3/mask'),
    'TNG 256³ (0.80)': ('tng300-3/n256/dm', 'tng300-3/n256/mask'),
    'SMDPL 512³ (0.78)': (SMDPL_DENSITY, SMDPL_LABELS),
}
SHIFTS = {'none': (0.0, 0.0, 0.0), 'x half': (0.5, 0.0, 0.0), 'diagonal half': (0.5, 0.5, 0.5)}   # cells

LAMBDA_TH = 0.65
SIGMA_EFF = 1.0          # h^-1 Mpc
MIN_RADIUS = 1.0         # as notebook 18
MARGIN = 96
R_CUT = 3.0
PERSISTENCE = [0.5, 1.0, 2.0]
DELTA_BINS = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, np.inf]      # bins of the unshifted join height, h^-1 Mpc
RATIO_BINS = [1.0, 1.05, 1.1, 1.25, 1.5, 2.0, np.inf]     # bins of the unshifted join ratio (peak / pass)
REFRESH = False


def timed(name, fn):
    # fn(), with how long the same step took last time (if known); records this run's time
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    print(f'{name}: ' + (f'last time {t[name] / 60:.1f} min' if name in t else 'no earlier timing'), flush=True)
    t0 = time.time()
    res = fn()
    dt = time.time() - t0
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    t[name] = dt
    TIMINGS.write_text(json.dumps(t, indent=1))
    print(f'{name}: {dt / 60:.1f} min', flush=True)
    return res

## 2 The runs

For each grid and placement: labels, distance, basins, merge tree, void counts. For a shifted placement, also the shifted basin under each unshifted basin's peak: the unshifted grid point *i* sits at *i* × cell and the shifted point *j* at (*j* + *s*) × cell, so the nearest shifted point is round(*i* − *s*) (for a half-cell shift, either neighbor is equally near; this takes *i*). A peak that lands on a shifted non-void voxel is counted as lost. The unshifted labels are compared with the labels already in use, as a check that they are the same labels.

In [ ]:
def density_of(src):
    if isinstance(src, Path):
        return Volume(data=np.load(src), box_size=SMDPL_BOX, meta={'sim': 'smdpl'})
    return catalog.load(src)


def labels_in_use(src):
    return (load_volume(src, dtype=None) if isinstance(src, Path) else catalog.load(src)).data


def one_run(grid, shift_name, ref_peaks=None):
    dens = density_of(GRIDS[grid][0])
    n, c = dens.data.shape[0], dens.box_size / dens.data.shape[0]
    lab = tweb_labels(dens, sigma_eff=SIGMA_EFF, threshold=LAMBDA_TH, shift=SHIFTS[shift_name]).data
    del dens
    gc.collect()
    out = {'n': n, 'cell': c, 'void_share': float((lab == VOID).mean())}
    if shift_name == 'none':
        try:
            old = labels_in_use(GRIDS[grid][1])
            out['agreement_with_labels_in_use'] = float((old == lab).mean())
            del old
        except Exception as err:
            print('could not read the labels in use:', repr(err))
    void = lab == VOID
    del lab
    d = V.distance_to_structure(void, cell=c, margin=MARGIN)
    seeds, nb = V.ascent_basins(d, void)
    t = V.merge_tree(seeds, d)
    out.update({k: np.asarray(v) for k, v in t.items()})
    out['n_basins'] = nb
    for p in PERSISTENCE:
        r = V.void_catalog(V.cut_tree(seeds, t, p, distance=d, min_radius=MIN_RADIUS), c)['r_eff']
        out[f'voids_p{p}'] = int((r > R_CUT).sum())
    if ref_peaks is not None:
        s = np.asarray(SHIFTS[shift_name])
        ijk = np.column_stack(np.unravel_index(ref_peaks, (n, n, n)))
        j = np.floor(ijk - s + 0.5).astype(np.int64) % n
        out['ref_peak_basin'] = seeds[j[:, 0], j[:, 1], j[:, 2]].astype(np.int64)
    del d, seeds, void
    gc.collect()
    return out


def run_path(grid, shift_name):
    return RUN_DIR / (f'{grid}_{shift_name}'.replace('³', '3').replace(' ', '_').replace('(', '').replace(')', '') + '.npz')


runs = {}
for grid in GRIDS:
    if isinstance(GRIDS[grid][0], Path) and not GRIDS[grid][0].exists():
        print(f'{grid}: {GRIDS[grid][0]} not found, skipped')
        continue
    for shift_name in SHIFTS:
        path = run_path(grid, shift_name)
        if path.exists() and not REFRESH:
            with np.load(path) as f:
                runs[grid, shift_name] = {k: f[k] for k in f.files}
            continue
        ref = None if shift_name == 'none' else runs[grid, 'none']['peak_index']
        res = timed(f'grid_noise_{grid}_{shift_name}', lambda: one_run(grid, shift_name, ref))
        np.savez(path, **res)
        runs[grid, shift_name] = res
    r0 = runs[grid, 'none']
    print(f"{grid}: {int(r0['n_basins'])} basins, void share {float(r0['void_share']):.3f}, agreement with the labels "
          f"in use {float(r0['agreement_with_labels_in_use']) if 'agreement_with_labels_in_use' in r0 else float('nan'):.4f}")

## 3 Join heights: unshifted against shifted

Per merge of the unshifted tree: the join height of its two void centers in the unshifted tree and in the shifted one, in Δ (h⁻¹ Mpc) and in the peak/pass ratio. "Lost" is the share of merges where a center's peak falls on a non-void voxel of the shifted grid; "same basin" the share where both centers fall in one shifted basin (that dent is not a separate basin there; its shifted join height is 0). Both are kept in the statistics. Pairs that never join in one of the trees (separate void regions) are left out.

The table gives, per bin of the unshifted join height, the median and 95th percentile of the absolute change, in h⁻¹ Mpc and in cells; both shifts are pooled.

In [ ]:
def heights(run, a, b, kind):
    w = run['persistence'] if kind == 'delta' else run['child_peak'] / np.maximum(run['pass_'], 1e-9)
    h = V.join_heights(len(run['peak']) - 1, run['child_edge'], run['parent_edge'], w, a, b)
    if kind == 'ratio':
        h = np.where(a == b, 1.0, h)          # one basin: no dip at all, ratio 1
    return h


def bin_label(bins, i):
    return f'{bins[i]:g}+' if np.isinf(bins[i + 1]) else f'{bins[i]:g}–{bins[i + 1]:g}'


rows, pairs, extra = [], {}, {}
for grid in GRIDS:
    if (grid, 'none') not in runs:
        continue
    r0 = runs[grid, 'none']
    cell = float(r0['cell'])
    a, b = r0['child'].astype(np.int64), r0['parent'].astype(np.int64)
    for kind, bins in (('delta', DELTA_BINS), ('ratio', RATIO_BINS)):
        h0_all, h1_all = [], []
        for shift_name in SHIFTS:
            if shift_name == 'none':
                continue
            r1 = runs[grid, shift_name]
            m = r1['ref_peak_basin']
            ma, mb = m[a], m[b]
            ok = (ma > 0) & (mb > 0)
            h0 = heights(r0, a[ok], b[ok], kind)
            h1 = heights(r1, ma[ok], mb[ok], kind)
            fin = np.isfinite(h0) & np.isfinite(h1)
            h0_all.append(h0[fin])
            h1_all.append(h1[fin])
            if kind == 'delta':
                extra[grid, shift_name] = {'merges': len(a), 'lost': float(1 - ok.mean()),
                                           'same basin': float((ma[ok] == mb[ok]).mean())}
        h0, h1 = np.concatenate(h0_all), np.concatenate(h1_all)
        pairs[grid, kind] = (h0, h1)
        change = np.abs(h1 - h0)
        idx = np.digitize(h0, bins[1:-1])
        for i in range(len(bins) - 1):
            sel = idx == i
            if not sel.any():
                continue
            row = {'grid': grid, 'measure': kind, 'bin': bin_label(bins, i), 'pairs': int(sel.sum()),
                   'median change': float(np.median(change[sel])), '95% change': float(np.quantile(change[sel], 0.95))}
            if kind == 'delta':
                row['median change (cells)'] = row['median change'] / cell
                row['95% change (cells)'] = row['95% change'] / cell
            rows.append(row)

table = pd.DataFrame(rows)
for kind in ('delta', 'ratio'):
    print({'delta': 'Join height Δ (h^-1 Mpc); change also in cells',
           'ratio': 'Join ratio (peak / pass); change in ratio units'}[kind])
    t = table[table.measure == kind].drop(columns='measure').dropna(axis=1, how='all')
    print(t.set_index(['grid', 'bin']).round(3).to_string(), '\n')
print(pd.DataFrame(extra).T.round(3).to_string())

## 4 Void counts and the figure

Voids with R_eff > `R_CUT` per grid and placement at each Δ, per 10⁶ (h⁻¹ Mpc)³. If the spread across placements is small next to the SMDPL–TNG difference, the placement of the grid points does not explain that difference (the cell size still could; compare TNG 256³ with TNG 512³).

Figure: left, the median (solid) and 95th percentile (dashed) of the change in join height, against the unshifted join height, one color per grid; middle, the same in cells; right, the same for the peak/pass ratio. Both half-cell shifts pooled.

In [ ]:
counts = {}
for (grid, shift_name), r in runs.items():
    vol = (float(r['n']) * float(r['cell'])) ** 3 / 1e6
    for p in PERSISTENCE:
        counts.setdefault((grid, f'Δ = {p}'), {})[shift_name] = int(r[f'voids_p{p}']) / vol
counts = pd.DataFrame(counts).T
counts['spread across placements'] = counts.max(axis=1) - counts.min(axis=1)
print('Voids above R_CUT per 10^6 (h^-1 Mpc)^3')
print(counts.round(2).to_string())

RESULT.write_text(json.dumps({'join_change': rows, 'lost_and_same_basin': {f'{g}|{s}': v for (g, s), v in extra.items()},
                              'voids_per_1e6': {f'{g}|{p}': v for (g, p), v in counts.to_dict('index').items()},
                              'settings': {'lambda_th': LAMBDA_TH, 'sigma_eff': SIGMA_EFF, 'min_radius': MIN_RADIUS,
                                           'r_cut': R_CUT, 'shifts': SHIFTS}}, indent=1))

colors = dict(zip(GRIDS, ['tab:blue', 'tab:orange', 'tab:green']))
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
panels = [('delta', 'median change', '95% change', 'change in join height Δ (h$^{-1}$ Mpc)', DELTA_BINS),
          ('delta', 'median change (cells)', '95% change (cells)', 'change in join height Δ (cells)', DELTA_BINS),
          ('ratio', 'median change', '95% change', 'change in join ratio (peak / pass)', RATIO_BINS)]
for ax, (kind, med, hi, ylab, bins) in zip(axes, panels):
    labels = [bin_label(bins, i) for i in range(len(bins) - 1)]
    for grid in colors:
        t = table[(table.grid == grid) & (table.measure == kind)].set_index('bin').reindex(labels)
        x = np.arange(len(labels))
        ax.plot(x, t[med], '-o', color=colors[grid], ms=4)
        ax.plot(x, t[hi], '--', color=colors[grid])
    ax.set_xticks(np.arange(len(labels)), labels, rotation=30)
    ax.set_xlabel('unshifted join height Δ (h$^{-1}$ Mpc)' if kind == 'delta' else 'unshifted join ratio (peak / pass)')
    ax.set_ylabel(ylab)
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(MaxNLocator(steps=[1, 2, 5, 10]))
    ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    ax.ticklabel_format(axis='y', style='plain')
axes[0].set_title('Join height: change after a half-cell shift')
axes[1].set_title('Same, in units of the cell')
axes[2].set_title('Peak / pass ratio: change after a half-cell shift')
handles = [plt.Line2D([], [], color=c, marker='o', ms=4, label=g) for g, c in colors.items()]
handles += [plt.Line2D([], [], color='gray', ls='-', label='median over merges'),
            plt.Line2D([], [], color='gray', ls='--', label='95th percentile')]
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=False,
           title='grid (cell, h$^{-1}$ Mpc)')
fig.suptitle('Grid noise in the T-web basin merge tree: each merge of the unshifted tree, '
             'its join height before and after moving the grid points by half a cell')
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(OUT / 'grid_noise.png', dpi=150, bbox_inches='tight')
plt.show()

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime